In [2]:
#import libraries

import pandas as pd
import numpy as np

In [9]:
#reading data

data = pd.read_csv("../Raw_Datasets/Diseases_LIMS_Combined.csv")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18600\1152101051.py:3: DtypeWarning: Columns (0: NOTIFIED TO) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("../Raw_Datasets/Diseases_LIMS_Combined.csv")


In [4]:
##view first 5 rows

data.head(5)

,ACCEPTING LAB,TESTING LAB,TESTING SECTION CODE,TESTING SECTION,SAMPLING PURPOSE,PLAN,SUBMITTER,NOTIFIED TO,SAMPLING POINT,OWNER,...,DISEASE,RESULT DATE,RESULT SENTENCE,SECOND RESULT SENTENCE,INTERPRETATION,TESTED BY,VALIDATED BY,NUM SAMPLES TESTED,DATE REPORTED,source_file
0,NVRL,NVRL,L1.6,Analytical Chemistry and Food Safety,Dip Strength Testing,EMPTY,NaN,NaN,NaN,NaN,...,EMPTY,2025-01-03,"Chlorpyrifos=288.36ppm, Cypermethrin=181.53ppm...",NaN,KO,NaN,NaN,1,2025-08-20 11:03:54,Diseases_LIMS_ January to March 2025.xls
1,NVRL,NVRL,L1.6,Analytical Chemistry and Food Safety,Dip Strength Testing,EMPTY,NaN,NaN,NaN,NaN,...,EMPTY,2025-01-03,"Chlorpyrifos=227.68ppm, Cypermethrin=166.51ppm...",NaN,KO,NaN,NaN,1,2025-08-20 11:03:54,Diseases_LIMS_ January to March 2025.xls
2,NVRL,NVRL,L1.6,Analytical Chemistry and Food Safety,Dip Strength Testing,EMPTY,NaN,NaN,NaN,NaN,...,EMPTY,2025-01-03,"Chlorpyrifos=215.58ppm, Cypermethrin=173.94ppm...",NaN,KO,NaN,NaN,1,2025-08-20 11:03:54,Diseases_LIMS_ January to March 2025.xls
3,KRT,KRT,L1.3.1,Culture and Identification,Passive AMR Surveillance,EMPTY,NaN,NaN,NaN,NaN,...,Bacterial Culture,2025-01-07,Bacterial growth,-Ba Small white and medium grey and yellow col...,OK,NaN,NaN,1,2025-01-07 15:49:26,Diseases_LIMS_ January to March 2025.xls
4,KRT,KRT,L1.3.1,Culture and Identification,Passive AMR Surveillance,EMPTY,NaN,NaN,NaN,NaN,...,Gram Stain technique results,2025-01-07,"Gram positive cocci in singles,pairs and clust...",NaN,OK,NaN,NaN,1,2025-01-07 15:49:26,Diseases_LIMS_ January to March 2025.xls


In [10]:
## shape
data.shape

(184952, 37)

In [13]:
# Convert date columns
date_cols = [
    "DATE RECEIVED",
    "SAMPLING DATE",
    "RECEIVED IN SECTION",
    "RESULT DATE",
    "DATE REPORTED"
]

for col in date_cols:
    data[col] = pd.to_datetime(data[col], errors="coerce", dayfirst=True)

# -----------------------------
#  Convert numeric columns
# -----------------------------
# Nullable integer because there are missing values
data["PRG.UNIT SAMPLE"] = data["PRG.UNIT SAMPLE"].astype("Int64")

# Integer columns with no missing values
data["SUBMISSION NUMBER"] = data["SUBMISSION NUMBER"].astype("int64")
data["NUM SAMPLES TESTED"] = data["NUM SAMPLES TESTED"].astype("int32")

# Columns that are completely empty → nullable string
empty_cols = [
    "SUBMITTER",
    "SAMPLING POINT",
    "OWNER",
    "TESTED BY",
    "VALIDATED BY"
]

for col in empty_cols:
    data[col] = data[col].astype("string")

# -----------------------------
# convert text columns to pandas string dtype
# -----------------------------
string_cols = [
    "NOTIFIED TO",
    "ACCEPTING LAB",
    "TESTING LAB",
    "TESTING SECTION CODE",
    "TESTING SECTION",
    "SAMPLING PURPOSE",
    "PLAN",
    "COUNTY",
    "SUB-COUNTY",
    "WARD",
    "SPECIES",
    "SAMPLE TYPE",
    "SAMPLE IDENTIFICATION",
    "SEX",
    "AGE",
    "TEST",
    "METHOD",
    "SPECIFIC TEST",
    "SOP",
    "DISEASE",
    "RESULT SENTENCE",
    "SECOND RESULT SENTENCE",
    "INTERPRETATION",
    "source_file"
]

for col in string_cols:
    data[col] = data[col].astype("string")

# -----------------------------
#  Convert repeated-value columns to category
# -----------------------------
category_cols = [
    "ACCEPTING LAB",
    "TESTING LAB",
    "TESTING SECTION CODE",
    "TESTING SECTION",
    "SAMPLING PURPOSE",
    "PLAN",
    "COUNTY",
    "SUB-COUNTY",
    "WARD",
    "SPECIES",
    "SAMPLE TYPE",
    "SEX",
    "TEST",
    "METHOD",
    "DISEASE"
]

for col in category_cols:
    data[col] = data[col].astype("category")

# Check the new structure
data.info()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18600\896818102.py:11: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  data[col] = pd.to_datetime(data[col], errors="coerce", dayfirst=True)


<class 'pandas.DataFrame'>
RangeIndex: 184952 entries, 0 to 184951
Data columns (total 37 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   ACCEPTING LAB           184952 non-null  category      
 1   TESTING LAB             184952 non-null  category      
 2   TESTING SECTION CODE    184952 non-null  category      
 3   TESTING SECTION         184952 non-null  category      
 4   SAMPLING PURPOSE        184952 non-null  category      
 5   PLAN                    184952 non-null  category      
 6   SUBMITTER               0 non-null       string        
 7   NOTIFIED TO             7867 non-null    string        
 8   SAMPLING POINT          0 non-null       string        
 9   OWNER                   0 non-null       string        
 10  SUBMISSION NUMBER       184952 non-null  int64         
 11  PRG.UNIT SAMPLE         184835 non-null  Int64         
 12  DATE RECEIVED           71061 non-null   

In [14]:
# Count and percentage of null values
null_summary = pd.DataFrame({
    "Null Count": data.isnull().sum(),
    "Null Percentage": (data.isnull().sum() / len(data) * 100).round(2)
})

# Sort by highest percentage of nulls
null_summary = null_summary.sort_values(by="Null Percentage", ascending=False)

print(null_summary)

                        Null Count  Null Percentage
TESTED BY                   184952           100.00
VALIDATED BY                184952           100.00
OWNER                       184952           100.00
SAMPLING POINT              184952           100.00
SUBMITTER                   184952           100.00
NOTIFIED TO                 177085            95.75
AGE                         173773            93.96
SEX                         171923            92.96
SOP                         152096            82.24
RECEIVED IN SECTION         118766            64.21
RESULT DATE                 116754            63.13
DATE RECEIVED               113891            61.58
SAMPLING DATE               113678            61.46
SECOND RESULT SENTENCE       80409            43.48
SPECIFIC TEST                79752            43.12
DATE REPORTED                12633             6.83
INTERPRETATION                3758             2.03
RESULT SENTENCE                192             0.10
PRG.UNIT SAM

In [15]:
# Drop columns that are completely null
data = data.dropna(axis=1, how="all")

# Fill missing values in text columns with "Unknown"
text_cols = data.select_dtypes(include=["object", "string"]).columns
data[text_cols] = data[text_cols].fillna("Unknown")

# Fill missing values in numeric columns with 0
num_cols = data.select_dtypes(include="number").columns
data[num_cols] = data[num_cols].fillna(0)

In [16]:
data.isnull().sum().sort_values(ascending=False)

SEX                       171923
RECEIVED IN SECTION       118766
RESULT DATE               116754
DATE RECEIVED             113891
SAMPLING DATE             113678
DATE REPORTED              12633
ACCEPTING LAB                  0
TESTING LAB                    0
SUBMISSION NUMBER              0
NOTIFIED TO                    0
PRG.UNIT SAMPLE                0
COUNTY                         0
SAMPLING PURPOSE               0
PLAN                           0
TESTING SECTION CODE           0
TESTING SECTION                0
SAMPLE TYPE                    0
SPECIES                        0
WARD                           0
SUB-COUNTY                     0
TEST                           0
METHOD                         0
AGE                            0
SAMPLE IDENTIFICATION          0
SOP                            0
SPECIFIC TEST                  0
RESULT SENTENCE                0
DISEASE                        0
SECOND RESULT SENTENCE         0
INTERPRETATION                 0
NUM SAMPLE

In [22]:
data1.columns


Index(['ACCEPTING LAB', 'TESTING LAB', 'TESTING SECTION CODE',
       'TESTING SECTION', 'SAMPLING PURPOSE', 'PLAN', 'NOTIFIED TO',
       'SUBMISSION NUMBER', 'PRG.UNIT SAMPLE', 'COUNTY', 'SUB-COUNTY', 'WARD',
       'SPECIES', 'SAMPLE TYPE', 'SAMPLE IDENTIFICATION', 'SEX', 'AGE', 'TEST',
       'METHOD', 'SPECIFIC TEST', 'SOP', 'DISEASE', 'RESULT SENTENCE',
       'SECOND RESULT SENTENCE', 'INTERPRETATION', 'NUM SAMPLES TESTED',
       'source_file'],
      dtype='str')

In [23]:
## drop irrelevant columns

data1 = data.drop(columns = ["RECEIVED IN SECTION","source_file","TESTING SECTION CODE","ACCEPTING LAB","TESTING LAB","RESULT DATE","DATE RECEIVED","SAMPLING DATE","DATE REPORTED"])

In [24]:
#view


data1.head(5)

,TESTING SECTION,SAMPLING PURPOSE,PLAN,NOTIFIED TO,SUBMISSION NUMBER,PRG.UNIT SAMPLE,COUNTY,SUB-COUNTY,WARD,SPECIES,...,AGE,TEST,METHOD,SPECIFIC TEST,SOP,DISEASE,RESULT SENTENCE,SECOND RESULT SENTENCE,INTERPRETATION,NUM SAMPLES TESTED
0,Analytical Chemistry and Food Safety,Dip Strength Testing,EMPTY,Unknown,1,1,NAIROBI,WESTLANDS,KITISURU,Any,...,Unknown,Dip Strength,GC,Unknown,Unknown,EMPTY,"Chlorpyrifos=288.36ppm, Cypermethrin=181.53ppm...",Unknown,KO,1
1,Analytical Chemistry and Food Safety,Dip Strength Testing,EMPTY,Unknown,1,2,NAIROBI,WESTLANDS,KITISURU,Any,...,Unknown,Dip Strength,GC,Unknown,Unknown,EMPTY,"Chlorpyrifos=227.68ppm, Cypermethrin=166.51ppm...",Unknown,KO,1
2,Analytical Chemistry and Food Safety,Dip Strength Testing,EMPTY,Unknown,1,3,NAIROBI,WESTLANDS,KITISURU,Any,...,Unknown,Dip Strength,GC,Unknown,Unknown,EMPTY,"Chlorpyrifos=215.58ppm, Cypermethrin=173.94ppm...",Unknown,KO,1
3,Culture and Identification,Passive AMR Surveillance,EMPTY,Unknown,2,1,KIRINYAGA,KIRINYAGA CENTRAL,KANYEKINI,Bovine,...,A,Bacterial culture,Isolation,Unknown,STANDARD OPERATING PROCEDURE OF SPECIMEN CULTU...,Bacterial Culture,Bacterial growth,-Ba Small white and medium grey and yellow col...,OK,1
4,Culture and Identification,Passive AMR Surveillance,EMPTY,Unknown,2,1,KIRINYAGA,KIRINYAGA CENTRAL,KANYEKINI,Bovine,...,Unknown,Gram stain Technique,Microscopical examination,Unknown,Unknown,Gram Stain technique results,"Gram positive cocci in singles,pairs and clust...",Unknown,OK,1


In [29]:
# Recode SEX values
data1["SEX"] = (
    data1["SEX"]
    .astype("string")
    .replace({
        "M": "Male",
        "F": "Female",
        "S": "Unknown",
        "N": "Unknown"
    })
    .fillna("Unknown")
)

In [30]:
data1.isnull().sum().sort_values(ascending=False)

TESTING SECTION           0
SAMPLING PURPOSE          0
PLAN                      0
NOTIFIED TO               0
SUBMISSION NUMBER         0
PRG.UNIT SAMPLE           0
COUNTY                    0
SUB-COUNTY                0
WARD                      0
SPECIES                   0
SAMPLE TYPE               0
SAMPLE IDENTIFICATION     0
SEX                       0
AGE                       0
TEST                      0
METHOD                    0
SPECIFIC TEST             0
SOP                       0
DISEASE                   0
RESULT SENTENCE           0
SECOND RESULT SENTENCE    0
INTERPRETATION            0
NUM SAMPLES TESTED        0
dtype: int64

In [32]:
data1.to_csv("../Cleaned Data/lims_cleaned.csv", index=False)